# Assignment 1: Sentence Classification with Deep Learning

This notebook covers all 5 tasks of Assignment 1, progressively building from a basic RNN to an attention-based BiGRU model.

| Task | Description |
|------|-------------|
| 1 | Configuration Optimization: packed sequences, regularization, hyperparameter search |
| 2 | Input Embedding: Pre-trained GloVe word vectors |
| 3 | Output Embedding: Mean pooling sentence representation |
| 4 | Architecture Optimization: BiGRU, BiLSTM, TextCNN |
| 5 | Critical Thinking: Attention mechanism |

In [ ]:
import torch
import numpy as np
import random
import time

import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:    
    device = torch.device('cpu')
    
print(f"Using device: {device}")

In [ ]:
# Install required packages
!pip install torchtext==0.4.0 -q
!python -m spacy download en_core_web_sm -q

In [ ]:
from torchtext import data, datasets

# TEXT field: tokenize with spaCy, keep sequence lengths for packed padding
TEXT = data.Field(
    tokenize='spacy',
    tokenizer_language='en_core_web_sm',
    include_lengths=True   # returns (tensor, lengths) for packed padded sequences
)

# LABEL field for multi-class classification
LABEL = data.LabelField()

In [ ]:
# Load TREC question classification dataset (coarse-grained, 6 classes)
train_data, test_data = datasets.TREC.splits(TEXT, LABEL, fine_grained=False)

# Create 80/20 train/validation split
train_data, valid_data = train_data.split(
    split_ratio=0.8,
    random_state=random.seed(SEED)
)

print(f'Training examples  : {len(train_data):,}')
print(f'Validation examples: {len(valid_data):,}')
print(f'Test examples      : {len(test_data):,}')
print(f'\nSample: {vars(train_data.examples[0])}')

In [ ]:
MAX_VOCAB_SIZE = 10_000
BATCH_SIZE     = 64

TEXT.build_vocab(train_data, max_size=MAX_VOCAB_SIZE)
LABEL.build_vocab(train_data)

VOCAB_SIZE   = len(TEXT.vocab)
OUTPUT_DIM   = len(LABEL.vocab)
PAD_IDX      = TEXT.vocab.stoi[TEXT.pad_token]
UNK_IDX      = TEXT.vocab.stoi[TEXT.unk_token]

print(f"TEXT  vocab size : {VOCAB_SIZE}")
print(f"LABEL vocab size : {OUTPUT_DIM}")
print(f"Labels           : {dict(LABEL.vocab.stoi)}")
print(f"PAD_IDX={PAD_IDX}, UNK_IDX={UNK_IDX}")

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data),
    batch_size=BATCH_SIZE,
    sort_within_batch=True,
    device=device
)

In [ ]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_epoch(model, iterator, optimizer, criterion):
    """One full pass over the training data."""
    epoch_loss, total_correct, total_instances = 0, 0, 0
    model.train()
    for batch in iterator:
        optimizer.zero_grad()
        text, text_lengths = batch.text
        predictions = model(text, text_lengths)
        loss = criterion(predictions, batch.label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
        _, pred_cls = predictions.max(dim=1)
        total_correct += (pred_cls == batch.label).sum().item()
        total_instances += batch.label.size(0)
    return epoch_loss / len(iterator), total_correct / total_instances


def evaluate_epoch(model, iterator, criterion):
    """Evaluate on a given iterator (no gradient computation)."""
    epoch_loss, total_correct, total_instances = 0, 0, 0
    model.eval()
    with torch.no_grad():
        for batch in iterator:
            text, text_lengths = batch.text
            predictions = model(text, text_lengths)
            loss = criterion(predictions, batch.label)
            epoch_loss += loss.item()
            _, pred_cls = predictions.max(dim=1)
            total_correct += (pred_cls == batch.label).sum().item()
            total_instances += batch.label.size(0)
    return epoch_loss / len(iterator), total_correct / total_instances


def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    return int(elapsed / 60), int(elapsed % 60)


def run_training(model, train_iter, valid_iter, optimizer, criterion,
                 n_epochs, save_path=None, verbose=True,
                 patience=None):
    """
    Train for up to n_epochs, saving the best model (lowest valid loss).

    Args:
        patience (int | None):
            Early-stopping patience. Training stops if valid_loss does not
            improve for this many consecutive epochs.
            Set to None (default) to disable early stopping.

    Returns:
        best_valid_loss  (float)
        best_valid_acc   (float)
        train_losses     (list[float])  -- per-epoch training loss
        val_losses       (list[float])  -- per-epoch validation loss
        epochs_run       (int)          -- actual number of epochs completed
    """
    best_valid_loss  = float('inf')
    best_valid_acc   = 0.0
    train_losses     = []
    val_losses       = []
    patience_counter = 0

    for epoch in range(n_epochs):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_iter, optimizer, criterion)
        vl_loss, vl_acc = evaluate_epoch(model, valid_iter, criterion)
        t1 = time.time()
        mins, secs = epoch_time(t0, t1)

        train_losses.append(round(tr_loss, 6))
        val_losses.append(round(vl_loss, 6))

        improved = vl_loss < best_valid_loss
        if improved:
            best_valid_loss  = vl_loss
            best_valid_acc   = vl_acc
            patience_counter = 0
            if save_path:
                torch.save(model.state_dict(), save_path)
        else:
            patience_counter += 1

        if verbose:
            patience_info = f'  [patience {patience_counter}/{patience}]' if patience else ''
            marker        = ' ✓ saved' if improved else ''
            print(f'Epoch {epoch+1:02}/{n_epochs} | {mins}m {secs}s{marker}{patience_info}')
            print(f'  Train Loss: {tr_loss:.4f}  Acc: {tr_acc*100:.2f}%')
            print(f'  Valid Loss: {vl_loss:.4f}  Acc: {vl_acc*100:.2f}%')

        # Early stopping check
        if patience is not None and patience_counter >= patience:
            if verbose:
                print(f'\n>> Early stopping triggered after {epoch+1} epochs')
                print(f'   (no val_loss improvement for {patience} consecutive epochs)')
            break

    epochs_run = len(train_losses)
    return best_valid_loss, best_valid_acc, train_losses, val_losses, epochs_run


---
## Task 1: Configuration Optimization

### 1a — Packed Padded Sequences

**Problem with the original implementation:** The baseline RNN processes ALL tokens including `<pad>` tokens equally. This means the final hidden state `h_T` is computed after seeing padding tokens, corrupting the sentence representation. For shorter sentences in a batch, the model may return a hidden state dominated by padding rather than the actual last word.

**Solution:** Use `pack_padded_sequence` + `pad_packed_sequence`. PyTorch skips padding tokens during the RNN computation, so each sentence's hidden state is extracted at its actual last real token. This:
- Gives more accurate sentence representations
- Is also faster (less computation on padding)

### 1b — Regularization

Two complementary techniques are added:
1. **Dropout** — applied after the embedding and before the final linear layer. Randomly zeroes activations during training, forcing the model to learn redundant representations and reducing co-adaptation.
2. **Weight Decay (L2 regularization)** — added to the optimizer. Penalises large weights, keeps the model from overfitting.
3. **Gradient Clipping** — already included in `run_training`; prevents the exploding gradient problem common in RNNs.

In [ ]:
class RNN_Task1(nn.Module):
    """
    Vanilla RNN with:
      - packed padded sequences  (Task 1a)
      - dropout regularisation   (Task 1b)
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.2, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))

        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(),
                                          enforce_sorted=False)
        packed_out, hidden = self.rnn(packed_emb)

        hidden = self.dropout(hidden.squeeze(0))
        # hidden        = [batch, hidden_dim]
        return self.fc(hidden)


print("RNN_Task1 model defined.")

### Task 1c — Hyperparameter Search

We search over: optimizer type, learning rate, hidden dimension, dropout rate, and weight decay.
Each configuration is trained for **5 epochs** on the validation set to efficiently compare. The best configuration is then trained for the full **10 epochs**.

In [ ]:
import itertools, os, json

DATA_DIR     = os.path.join('.data')
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = os.path.join(DATA_DIR, 'Task1_HyperparameterTuning.json')

# Load existing results so a crashed run can be resumed without re-running configs
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as _f:
        all_results = json.load(_f)
    print(f'Loaded {len(all_results)} existing result(s) from {RESULTS_FILE}')
else:
    all_results = []
    print(f'Starting fresh — results will be saved to: {RESULTS_FILE}')

already_run = {r['config']['name'] for r in all_results}

# ── Search grid ────────────────────────────────────────────────────────────────
SEARCH_EPOCHS   = 20
EARLY_STOP_PAT  = 3          # stop a config if no val improvement for 5 epochs

opt_configs     = ['SGD', 'Adam']
lr_configs      = [0.01, 1e-3, 1e-4]
hidden_configs  = [50, 100, 256, 512]
dropout_configs = [0.0, 0.2, 0.3, 0.4]
wd_configs      = [0, 1e-4]

criterion    = nn.CrossEntropyLoss().to(device)
search_results = []   # results from THIS run (not including resumed ones)

total = (len(opt_configs) * len(lr_configs) *
         len(hidden_configs) * len(dropout_configs) * len(wd_configs))
done  = 0

for (opt_type, lr, hid_dim, dropout, wd) in itertools.product(
        opt_configs, lr_configs, hidden_configs, dropout_configs, wd_configs):

    done       += 1
    config_name = f'{opt_type}_lr={lr}_hid={hid_dim}_drop={dropout}_wd={wd}'

    # Skip configs already persisted (resume support)
    if config_name in already_run:
        print(f'[{done:>4}/{total}] SKIP (already saved): {config_name}')
        continue

    print(f'\n[{done:>4}/{total}] {config_name}')

    torch.manual_seed(SEED)
    m = RNN_Task1(VOCAB_SIZE, 100, hid_dim, OUTPUT_DIM,
                  dropout=dropout, pad_idx=PAD_IDX).to(device)

    if opt_type == 'Adam':
        opt = optim.Adam(m.parameters(), lr=lr, weight_decay=wd)
    else:
        opt = optim.SGD(m.parameters(), lr=lr, weight_decay=wd)

    # Train — early stopping enabled
    best_vl, best_vacc, tr_losses, vl_losses, epochs_run = run_training(
        m, train_iterator, valid_iterator, opt, criterion,
        n_epochs   = SEARCH_EPOCHS,
        save_path  = None,
        verbose    = False,
        patience   = EARLY_STOP_PAT
    )

    stopped_early = epochs_run < SEARCH_EPOCHS
    tag = '  (early stop)' if stopped_early else ''
    print(f'   epochs={epochs_run:>2} | val_acc={best_vacc*100:.2f}%'
          f' | val_loss={best_vl:.4f}{tag}')

    # ── Build result record ─────────────────────────────────────────────────────
    result = {
        'config': {
            'name'      : config_name,
            'opt_type'  : opt_type,
            'lr'        : lr,
            'hidden_dim': hid_dim,
            'dropout'   : dropout,
            'wd'        : wd,
        },
        'num_epochs'     : epochs_run,
        'stopped_early'  : stopped_early,
        'best_valid_loss': round(best_vl,    6),
        'best_valid_acc' : round(best_vacc,  6),
        'train_losses'   : tr_losses,
        'val_losses'     : vl_losses,
    }

    all_results.append(result)
    search_results.append(result)

    # Persist after every config — safe against mid-run crashes
    with open(RESULTS_FILE, 'w') as _f:
        json.dump(all_results, _f, indent=2)

print(f'\nAll done. {len(search_results)} new config(s) evaluated.')
print(f'Results saved to: {RESULTS_FILE}')

# ── Pick best across ALL results (including any resumed ones) ───────────────────
best_row  = max(all_results, key=lambda r: r['best_valid_acc'])
BEST_NAME = best_row['config']['name']
BEST_OPT  = best_row['config']['opt_type']
BEST_LR   = best_row['config']['lr']
BEST_EMB  = 100
BEST_HID  = best_row['config']['hidden_dim']
BEST_DROP = best_row['config']['dropout']
BEST_WD   = best_row['config']['wd']

print(f"\n{'='*60}")
print(f"Best config  : {BEST_NAME}")
print(f"Valid Acc    : {best_row['best_valid_acc']*100:.2f}%")
print(f"Epochs run   : {best_row['num_epochs']}")
print(f"Results file : {RESULTS_FILE}")
print(f"{'='*60}")


In [ ]:
N_EPOCHS = 100

torch.manual_seed(SEED)
model_t1 = RNN_Task1(VOCAB_SIZE, BEST_EMB, BEST_HID, OUTPUT_DIM,
                     dropout=BEST_DROP, pad_idx=PAD_IDX).to(device)

if BEST_OPT == 'Adam':
    opt_t1 = optim.Adam(model_t1.parameters(), lr=BEST_LR, weight_decay=BEST_WD)
else:
    opt_t1 = optim.SGD(model_t1.parameters(), lr=BEST_LR, weight_decay=BEST_WD)

print(f"Task 1 model — {count_parameters(model_t1):,} trainable parameters")
print(f"Best config: {BEST_NAME}\n")

run_training(model_t1, train_iterator, valid_iterator, opt_t1, criterion,
             N_EPOCHS, save_path='models/task1-model.pt', patience= EARLY_STOP_PAT)

In [ ]:
model_t1.load_state_dict(torch.load('models/task1-model.pt'))

_, valid_acc_t1 = evaluate_epoch(model_t1, valid_iterator, criterion)
_, test_acc_t1  = evaluate_epoch(model_t1, test_iterator,  criterion)

print("=" * 50)
print("Task 1 Results (RNN + Packed Seq + Dropout + Best Hyper)")
print(f"  Valid Acc : {valid_acc_t1 *100:.2f}%")
print(f"  Test  Acc : {test_acc_t1 *100:.2f}%")
print("=" * 50)

---
## Task 2: Pre-trained Input Embeddings (FastText)

We replace randomly initialised word embeddings with **FastText English 300-dimensional** vectors.

**Why FastText over GloVe or Word2Vec?**
FastText (Bojanowski et al., 2017) extends Word2Vec by representing each word as a bag of
character n-grams. The embedding for a word is the sum of its subword vectors, which gives
three key advantages for this task:

- **OOV handling** — words not seen during FastText training still get a meaningful vector
  composed from their character n-grams, unlike GloVe which falls back to a random vector.
- **Morphological awareness** — question words share subwords (e.g. *"abbreviate"*,
  *"abbreviation"*, *"abbreviated"*), so FastText transfers morphological knowledge that
  GloVe treats as independent entries.
- **TREC coverage** — TREC questions contain proper nouns, rare verbs and numerical
  expressions that are underrepresented in GloVe's vocabulary but recoverable via subwords.

The `<unk>` and `<pad>` token embeddings are zeroed out after initialisation to avoid
spurious signals. All parameters including the embeddings are fine-tuned during training.

In [ ]:
from torchtext import data, datasets

# TEXT field: tokenize with spaCy, keep sequence lengths for packed padding
TEXT = data.Field(
    tokenize='spacy',
    tokenizer_language='en_core_web_sm',
    include_lengths=True   # returns (tensor, lengths) for packed padded sequences
)

# LABEL field for multi-class classification
LABEL = data.LabelField()

In [ ]:
# Load TREC question classification dataset (coarse-grained, 6 classes)
train_data, test_data = datasets.TREC.splits(TEXT, LABEL, fine_grained=False)

# Create 80/20 train/validation split
train_data, valid_data = train_data.split(
    split_ratio=0.8,
    random_state=random.seed(SEED)
)

print(f'Training examples  : {len(train_data):,}')
print(f'Validation examples: {len(valid_data):,}')
print(f'Test examples      : {len(test_data):,}')
print(f'\nSample: {vars(train_data.examples[0])}')

In [ ]:
from torchtext.vocab import FastText

MAX_VOCAB_SIZE = 20000
BATCH_SIZE     = 64

# Rebuild vocabulary with FastText English 300-d vectors
# FastText uses subword n-grams so OOV words still get meaningful vectors
print("Loading FastText English 300d vectors (download ~2.6 GB on first run)...")
try:
    ft_vectors = FastText(language='en')
    TEXT.build_vocab(
        train_data,
        max_size=MAX_VOCAB_SIZE,
        vectors=ft_vectors,
        unk_init=torch.Tensor.normal_  # random normal for OOV words
    )
    EMBEDDING_DIM = 300   # FastText vectors are 300-dimensional
    LABEL.build_vocab(train_data)
    print(f"FastText loaded. Vector matrix: {TEXT.vocab.vectors.shape}")
except Exception as e:
    print(f"FastText download failed ({e}). Falling back to random init.")
    TEXT.build_vocab(train_data, max_size=MAX_VOCAB_SIZE)
    EMBEDDING_DIM = 100

VOCAB_SIZE = len(TEXT.vocab)
PAD_IDX    = TEXT.vocab.stoi[TEXT.pad_token]
UNK_IDX    = TEXT.vocab.stoi[TEXT.unk_token]

train_iterator, valid_iterator, test_iterator = data.BucketIterator.splits(
    (train_data, valid_data, test_data),
    batch_size=BATCH_SIZE,
    sort_within_batch=True,
    device=device
)

print(f"Vocab size    : {VOCAB_SIZE}")
print(f"Embedding dim : {EMBEDDING_DIM}")

In [ ]:
import itertools, os, json
# ── Directory & results file ───────────────────────────────────────────────────
DATA_DIR        = '.data'
os.makedirs(DATA_DIR, exist_ok=True)
T2_RESULTS_FILE = os.path.join(DATA_DIR, 'Task2_HyperparameterTuning.json')

# Resume support — load any configs already completed
if os.path.exists(T2_RESULTS_FILE):
    with open(T2_RESULTS_FILE) as _f:
        t2_all_results = json.load(_f)
    print(f"Resuming — {len(t2_all_results)} config(s) already saved.")
else:
    t2_all_results = []
    print(f"Starting fresh — results will be saved to: {T2_RESULTS_FILE}")

t2_already_run = {r['config']['name'] for r in t2_all_results}

# ── Reduced search grid (informed by Task 1 findings) ─────────────────────────
# Task 1 showed Adam consistently outperforms SGD on this dataset,
# so we restrict to Adam only. WD=1e-4 was best in Task 1.
T2_SEARCH_EPOCHS   = 20
T2_EARLY_STOP_PAT  = 5
OUTPUT_DIM          = len(LABEL.vocab)  # 6 classes

t2_lr_configs      = [0.01, 1e-3, 1e-4]
t2_hidden_configs  = [50, 100, 256, 512]
t2_dropout_configs = [0.2, 0.3, 0.4]
t2_wd              = 1e-4          # fixed — best from Task 1
t2_opt             = 'Adam'        # fixed — best from Task 1

total_t2 = len(t2_lr_configs) * len(t2_hidden_configs) * len(t2_dropout_configs)
print(f"Grid size: {total_t2} configs  "
      f"(Adam only, wd={t2_wd}, "
      f"lr={t2_lr_configs}, "
      f"hidden={t2_hidden_configs}, "
      f"dropout={t2_dropout_configs})")

criterion = nn.CrossEntropyLoss().to(device)
t2_search_results = []
done = 0

for (lr, hid_dim, dropout) in itertools.product(
        t2_lr_configs, t2_hidden_configs, t2_dropout_configs):

    done += 1
    config_name = f'Adam_lr={lr}_hid={hid_dim}_drop={dropout}_wd={t2_wd}'

    if config_name in t2_already_run:
        print(f'[{done:>3}/{total_t2}] SKIP: {config_name}')
        continue

    print(f'\n[{done:>3}/{total_t2}] {config_name}')

    torch.manual_seed(SEED)
    m = RNN_Task1(VOCAB_SIZE, EMBEDDING_DIM, hid_dim, OUTPUT_DIM,
                  dropout=dropout, pad_idx=PAD_IDX).to(device)

    # Initialise with FastText vectors
    m.embedding.weight.data.copy_(TEXT.vocab.vectors)
    m.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    m.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

    opt = optim.Adam(m.parameters(), lr=lr, weight_decay=t2_wd)

    best_vl, best_vacc, tr_losses, vl_losses, epochs_run = run_training(
        m, train_iterator, valid_iterator, opt, criterion,
        n_epochs   = T2_SEARCH_EPOCHS,
        save_path  = None,
        verbose    = False,
        patience   = T2_EARLY_STOP_PAT
    )

    stopped_early = epochs_run < T2_SEARCH_EPOCHS
    tag = '  (early stop)' if stopped_early else ''
    print(f'   epochs={epochs_run:>2} | val_acc={best_vacc*100:.2f}%'
          f' | val_loss={best_vl:.4f}{tag}')

    result = {
        'config': {
            'name'      : config_name,
            'opt_type'  : t2_opt,
            'lr'        : lr,
            'hidden_dim': hid_dim,
            'dropout'   : dropout,
            'wd'        : t2_wd,
            'embedding' : 'FastText-en-300d',
        },
        'num_epochs'     : epochs_run,
        'stopped_early'  : stopped_early,
        'best_valid_loss': round(best_vl,   6),
        'best_valid_acc' : round(best_vacc, 6),
        'train_losses'   : tr_losses,
        'val_losses'     : vl_losses,
    }

    t2_all_results.append(result)
    t2_search_results.append(result)

    # Persist after every config
    with open(T2_RESULTS_FILE, 'w') as _f:
        json.dump(t2_all_results, _f, indent=2)

print(f'\nDone. {len(t2_search_results)} new config(s) evaluated.')
print(f'Results saved → {T2_RESULTS_FILE}')

# ── Best config ────────────────────────────────────────────────────────────────
t2_best_row  = max(t2_all_results, key=lambda r: r['best_valid_acc'])
T2_BEST_NAME = t2_best_row['config']['name']
T2_BEST_LR   = t2_best_row['config']['lr']
T2_BEST_HID  = t2_best_row['config']['hidden_dim']
T2_BEST_DROP = t2_best_row['config']['dropout']
T2_BEST_WD   = t2_best_row['config']['wd']

print(f"\n{'='*60}")
print(f"Best Task 2 config : {T2_BEST_NAME}")
print(f"Val Acc            : {t2_best_row['best_valid_acc']*100:.2f}%")
print(f"{'='*60}")

In [ ]:
# ── Retrain best config for full N_EPOCHS ─────────────────────────────────────
print(f"Retraining best Task 2 config: {T2_BEST_NAME}\n")
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)   

torch.manual_seed(SEED)
model_t2 = RNN_Task1(VOCAB_SIZE, EMBEDDING_DIM, T2_BEST_HID, OUTPUT_DIM,
                     dropout=T2_BEST_DROP, pad_idx=PAD_IDX).to(device)

# Initialise with FastText vectors
model_t2.embedding.weight.data.copy_(TEXT.vocab.vectors)
model_t2.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
model_t2.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)

opt_t2 = optim.Adam(model_t2.parameters(), lr=T2_BEST_LR, weight_decay=T2_BEST_WD)

print(f"Task 2 model — {count_parameters(model_t2):,} trainable parameters")

_, _, t2_train_losses, t2_val_losses, _ = run_training(
    model_t2, train_iterator, valid_iterator, opt_t2, criterion,
    n_epochs   = 50,
    save_path  = os.path.join(MODEL_DIR, 'task2-best-model.pt'),
    patience   = T2_EARLY_STOP_PAT
)

# ── Evaluate ───────────────────────────────────────────────────────────────────
model_t2.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'task2-best-model.pt')))

_, valid_acc_t2 = evaluate_epoch(model_t2, valid_iterator, criterion)
_, test_acc_t2  = evaluate_epoch(model_t2, test_iterator,  criterion)

print("=" * 60)
print("Task 2 Results (FastText 300d  |  best config from grid search)")
print(f"  Valid Acc : {valid_acc_t2*100:.2f}% ")
print(f"  Test  Acc : {test_acc_t2 *100:.2f}% ")
print("=" * 60)

---
## Task 3: Output Pooling Strategies

The baseline RNN uses only the **final hidden state** `h_T` as the sentence representation.
This forces the entire sentence meaning into a single vector at the last timestep, which can
discard information from earlier tokens — especially when key classification words appear
mid-sentence.

We evaluate three pooling strategies over all non-padded hidden states `{h_1, …, h_T}`,
each applied to the **same RNN architecture** trained with the **best Task 2 hyperparameters**
(FastText 300d embeddings, Adam, no additional grid search):

| Strategy | Sentence vector | Intuition |
|---|---|---|
| **Mean Pooling** | `(1/T) Σ hₜ` | Every token contributes equally |
| **Max Pooling** | `max_t hₜ` (element-wise) | Captures the most salient feature per dimension |
| **Attention Pooling** | `Σ αₜ hₜ`, `αₜ = softmax(w·hₜ)` | Learned, input-dependent weighting of positions |


In [ ]:
# ── Task 3: Three pooling strategy model definitions ──────────────────────────

class RNN_MeanPool(nn.Module):
    """Mean pooling over non-padded hidden states."""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(), enforce_sorted=False)
        packed_out, _ = self.rnn(packed_emb)
        output, out_lengths = pad_packed_sequence(packed_out)   # [T, B, H]
        output = output.permute(1, 0, 2)                        # [B, T, H]

        lengths_f = out_lengths.to(device).float()
        mask = (torch.arange(output.size(1), device=device)
                     .unsqueeze(0) < lengths_f.long().unsqueeze(1))  # [B, T]
        masked_out  = output * mask.unsqueeze(2).float()
        mean_pooled = masked_out.sum(dim=1) / lengths_f.unsqueeze(1) # [B, H]
        return self.fc(self.dropout(mean_pooled))


class RNN_MaxPool(nn.Module):
    """Element-wise max pooling over non-padded hidden states."""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(), enforce_sorted=False)
        packed_out, _ = self.rnn(packed_emb)
        output, out_lengths = pad_packed_sequence(packed_out)   # [T, B, H]
        output = output.permute(1, 0, 2)                        # [B, T, H]

        # Mask padding with -inf before max so pad positions are never selected
        mask = (torch.arange(output.size(1), device=device)
                     .unsqueeze(0) >= out_lengths.to(device).long().unsqueeze(1))  # [B, T]
        output = output.masked_fill(mask.unsqueeze(2), float('-inf'))
        max_pooled, _ = output.max(dim=1)                       # [B, H]
        return self.fc(self.dropout(max_pooled))


class RNN_AttnPool(nn.Module):
    """Additive (Bahdanau-style) self-attention pooling over hidden states."""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 dropout=0.5, pad_idx=None):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.rnn       = nn.RNN(embedding_dim, hidden_dim, batch_first=False)
        self.attn      = nn.Linear(hidden_dim, 1, bias=False)  # scoring vector
        self.fc        = nn.Linear(hidden_dim, output_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))
        packed_emb = pack_padded_sequence(embedded, text_lengths.cpu(), enforce_sorted=False)
        packed_out, _ = self.rnn(packed_emb)
        output, out_lengths = pad_packed_sequence(packed_out)   # [T, B, H]
        output = output.permute(1, 0, 2)                        # [B, T, H]

        # Attention scores: e_t = w · h_t
        energy = self.attn(output).squeeze(2)                   # [B, T]

        # Mask padding before softmax
        mask = (torch.arange(output.size(1), device=device)
                     .unsqueeze(0) >= out_lengths.to(device).long().unsqueeze(1))  # [B, T]
        energy = energy.masked_fill(mask, float('-inf'))

        attn_weights = torch.softmax(energy, dim=1)             # [B, T]
        attn_out = (output * attn_weights.unsqueeze(2)).sum(dim=1)  # [B, H]
        return self.fc(self.dropout(attn_out))


In [ ]:
# ── Task 3: Train all three pooling models with Task 2 best hyperparameters ───
# No new grid search — use T2_BEST_* variables from Task 2 cell above.

MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

def make_t3_model(ModelClass):
    """Instantiate a T3 model, initialise with FastText vectors, return (model, optimizer)."""
    torch.manual_seed(SEED)
    m = ModelClass(VOCAB_SIZE, EMBEDDING_DIM, T2_BEST_HID, OUTPUT_DIM,
                   dropout=T2_BEST_DROP, pad_idx=PAD_IDX).to(device)
    m.embedding.weight.data.copy_(TEXT.vocab.vectors)
    m.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    m.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)
    opt = optim.Adam(m.parameters(), lr=T2_BEST_LR, weight_decay=T2_BEST_WD)
    return m, opt

T3_CONFIGS = [
    ('MeanPool',  RNN_MeanPool,  'task3-mean-model.pt'),
    ('MaxPool',   RNN_MaxPool,   'task3-max-model.pt'),
    ('AttnPool',  RNN_AttnPool,  'task3-attn-model.pt'),
]

t3_models = {}

for name, ModelClass, save_name in T3_CONFIGS:
    save_path = os.path.join(MODEL_DIR, save_name)
    print(f"\n{'='*55}")
    print(f"Training Task 3 — {name}")
    print(f"  hid={T2_BEST_HID}, lr={T2_BEST_LR}, drop={T2_BEST_DROP}, wd={T2_BEST_WD}")
    print(f"{'='*55}")

    m, opt = make_t3_model(ModelClass)
    print(f"  Parameters: {count_parameters(m):,}")

    run_training(m, train_iterator, valid_iterator, opt, criterion,
                 n_epochs=50, save_path=save_path, patience=5)

    t3_models[name] = (m, save_path)

print("\nAll Task 3 models trained.")


In [ ]:
# ── Task 3: Evaluate all three pooling strategies ────────────────────────────

T3_CLASSES = {
    'MeanPool' : RNN_MeanPool,
    'MaxPool'  : RNN_MaxPool,
    'AttnPool' : RNN_AttnPool,
}
T3_SAVES = {
    'MeanPool' : 'task3-mean-model.pt',
    'MaxPool'  : 'task3-max-model.pt',
    'AttnPool' : 'task3-attn-model.pt',
}

t3_results = {}

for name, ModelClass in T3_CLASSES.items():
    m, _ = make_t3_model(ModelClass)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, T3_SAVES[name])))
    _, v_acc = evaluate_epoch(m, valid_iterator, criterion)
    _, te_acc = evaluate_epoch(m, test_iterator,  criterion)
    t3_results[name] = {'valid': v_acc, 'test': te_acc}

# Store best Task 3 result (for Task 4 baseline comparison)
best_t3_name = max(t3_results, key=lambda k: t3_results[k]['valid'])
valid_acc_t3  = t3_results[best_t3_name]['valid']
test_acc_t3   = t3_results[best_t3_name]['test']

# ── Print comparison table ────────────────────────────────────────────────────
print("=" * 65)
print("Task 3 — Pooling Strategy Comparison  (FastText 300d, T2 best params)")
print("-" * 65)
print(f"{'Model':<25} {'Valid Acc':>10} {'Test Acc':>10} {'ΔValid vs T2':>14}")
print("-" * 65)
print(f"{'Task 2  (final h_T)':<25} {valid_acc_t2*100:>9.2f}% {test_acc_t2*100:>9.2f}% {'baseline':>14}")
for name in ['MeanPool', 'MaxPool', 'AttnPool']:
    r = t3_results[name]
    delta = (r['valid'] - valid_acc_t2) * 100
    marker = ' ◀ best' if name == best_t3_name else ''
    print(f"{'Task 3  '+name:<25} {r['valid']*100:>9.2f}% {r['test']*100:>9.2f}% {delta:>+13.2f}%{marker}")
print("=" * 65)


---
## Task 4: Architecture Optimization with Grid Search

We replace the unidirectional RNN with three advanced architectures, each trained with:
- **FastText 300d pre-trained embeddings** (fine-tuned during training)
- The **same grid search as Task 2**: Adam, WD=1e-4, lr ∈ {0.01, 1e-3, 1e-4}, hidden ∈ {50, 100, 256, 512}, dropout ∈ {0.2, 0.3, 0.4}
- **Three output pooling strategies per architecture**: `last` (final hidden state), `mean` (mean over time), `max` (max over time) — giving 108 configs per BiRNN and 72 for TextCNN (no `last` for CNN)

Results are saved per-architecture to `.data/` with resume support.

| Architecture | Key property | Pooling options |
|---|---|---|
| **BiGRU** | Bidirectional GRU, 2 layers | last / mean / max |
| **BiLSTM** | Bidirectional LSTM, 2 layers | last / mean / max |
| **TextCNN** | Parallel n-gram convolutions | max / mean |


In [ ]:
# ── Task 4: Model Definitions ─────────────────────────────────────────────────

class BiRNN(nn.Module):
    """
    Bidirectional GRU or LSTM with configurable output pooling (Task 4).
    pooling: 'last'  → concat final fwd/bwd hidden states of top layer
             'mean'  → masked mean over all timesteps
             'max'   → masked element-wise max over all timesteps
    """
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim,
                 n_layers=2, dropout=0.5, pad_idx=None, rnn_type='GRU', pooling='last'):
        super().__init__()
        self.pooling  = pooling
        self.rnn_type = rnn_type
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        RNNClass = nn.GRU if rnn_type == 'GRU' else nn.LSTM
        self.rnn = RNNClass(
            embedding_dim, hidden_dim,
            num_layers=n_layers, bidirectional=True,
            dropout=(dropout if n_layers > 1 else 0.0),
            batch_first=False
        )
        self.fc      = nn.Linear(hidden_dim * 2, output_dim)   # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        embedded = self.dropout(self.embedding(text))           # [T, B, E]
        packed   = pack_padded_sequence(embedded, text_lengths.cpu(), enforce_sorted=False)

        if self.rnn_type == 'GRU':
            packed_out, hidden = self.rnn(packed)
            # hidden: [n_layers*2, B, H] — index -2 = top fwd, -1 = top bwd
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)   # [B, 2H]
        else:   # LSTM
            packed_out, (hidden, _) = self.rnn(packed)
            last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)   # [B, 2H]

        output, out_lengths = pad_packed_sequence(packed_out)  # [T, B, 2H]
        output = output.permute(1, 0, 2)                       # [B, T, 2H]

        if self.pooling == 'last':
            sentence = last_hidden

        elif self.pooling == 'mean':
            lengths_f = out_lengths.to(device).float()
            mask = (torch.arange(output.size(1), device=device)
                         .unsqueeze(0) < lengths_f.long().unsqueeze(1))  # [B, T]
            sentence = (output * mask.unsqueeze(2).float()).sum(dim=1) / lengths_f.unsqueeze(1)

        elif self.pooling == 'max':
            pad_mask = (torch.arange(output.size(1), device=device)
                             .unsqueeze(0) >= out_lengths.to(device).long().unsqueeze(1))
            sentence, _ = output.masked_fill(pad_mask.unsqueeze(2), float('-inf')).max(dim=1)

        return self.fc(self.dropout(sentence))


class TextCNN(nn.Module):
    """
    Kim (2014)-style TextCNN with configurable pooling (Task 4).
    pooling: 'max'  → max-over-time per filter (standard)
             'mean' → mean-over-time per filter
    """
    FILTER_SIZES = [2, 3, 4]

    def __init__(self, vocab_size, embedding_dim, n_filters, output_dim,
                 dropout=0.5, pad_idx=None, pooling='max'):
        super().__init__()
        self.pooling   = pooling
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.convs     = nn.ModuleList([
            nn.Conv2d(1, n_filters, (fs, embedding_dim))
            for fs in self.FILTER_SIZES
        ])
        self.fc      = nn.Linear(n_filters * len(self.FILTER_SIZES), output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, text, text_lengths):
        text     = text.permute(1, 0)                          # [B, T]
        embedded = self.dropout(self.embedding(text))          # [B, T, E]
        embedded = embedded.unsqueeze(1)                       # [B, 1, T, E]

        # Each conv: [B, 1, T, E] -> [B, n_filters, T-fs+1, 1] -> squeeze -> [B, n_f, T-fs+1]
        conved = [torch.relu(conv(embedded)).squeeze(3) for conv in self.convs]

        if self.pooling == 'max':
            pooled = [c.max(dim=2)[0] for c in conved]        # each [B, n_filters]
        else:   # mean
            pooled = [c.mean(dim=2) for c in conved]

        cat = self.dropout(torch.cat(pooled, dim=1))           # [B, n_filters * 3]
        return self.fc(cat)


print("BiRNN and TextCNN model classes defined.")
print(f"  BiRNN pooling options : last / mean / max")
print(f"  TextCNN pooling options: max / mean")


In [ ]:
# ── Task 4: Generic grid search + FastText init helper ────────────────────────

def init_ft_embeddings(model):
    """Copy FastText 300d vectors into model.embedding; zero UNK and PAD."""
    model.embedding.weight.data.copy_(TEXT.vocab.vectors)
    model.embedding.weight.data[UNK_IDX] = torch.zeros(EMBEDDING_DIM)
    model.embedding.weight.data[PAD_IDX]  = torch.zeros(EMBEDDING_DIM)


def t4_grid_search(arch_name, build_model_fn, pooling_options, results_file,
                   lr_list=None, hidden_list=None, dropout_list=None,
                   wd=1e-4, search_epochs=20, patience=5):
    """
    Generic Task 4 grid search with resume support.

    build_model_fn(hidden_dim, dropout, pooling) -> nn.Module (on CPU, not yet on device)
    Results are persisted to results_file after every config.
    Returns (all_results, best_row).
    """
    if lr_list is None:      lr_list      = [0.01, 1e-3, 1e-4]
    if hidden_list is None:  hidden_list  = [50, 100, 256, 512]
    if dropout_list is None: dropout_list = [0.2, 0.3, 0.4]

    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(results_file):
        with open(results_file) as _f:
            all_results = json.load(_f)
        print(f"Resuming — {len(all_results)} config(s) already done.")
    else:
        all_results = []
    already_run = {r['config']['name'] for r in all_results}

    total = len(pooling_options) * len(lr_list) * len(hidden_list) * len(dropout_list)
    print(f"\n{arch_name} grid: {total} configs")
    print(f"  pooling={pooling_options}  lr={lr_list}")
    print(f"  hidden={hidden_list}  dropout={dropout_list}  wd={wd}")

    done = 0
    for (pool, lr, hid, drop) in itertools.product(
            pooling_options, lr_list, hidden_list, dropout_list):
        done += 1
        name = f'{arch_name}_pool={pool}_lr={lr}_hid={hid}_drop={drop}_wd={wd}'
        if name in already_run:
            print(f'[{done:>3}/{total}] SKIP: {name}')
            continue
        print(f'\n[{done:>3}/{total}] {name}')

        torch.manual_seed(SEED)
        m = build_model_fn(hid, drop, pool).to(device)
        init_ft_embeddings(m)
        opt = optim.Adam(m.parameters(), lr=lr, weight_decay=wd)

        best_vl, best_vacc, tr_losses, vl_losses, epochs_run = run_training(
            m, train_iterator, valid_iterator, opt, criterion,
            n_epochs=search_epochs, save_path=None,
            verbose=False, patience=patience
        )
        stopped = epochs_run < search_epochs
        print(f'   epochs={epochs_run:>2} | val_acc={best_vacc*100:.2f}%'
              f' | val_loss={best_vl:.4f}{"  (early stop)" if stopped else ""}')

        result = {
            'config': {
                'name': name, 'arch': arch_name, 'pooling': pool,
                'lr': lr, 'hidden_dim': hid, 'dropout': drop, 'wd': wd,
                'embedding': 'FastText-en-300d',
            },
            'num_epochs': epochs_run, 'stopped_early': stopped,
            'best_valid_loss': round(best_vl, 6),
            'best_valid_acc':  round(best_vacc, 6),
            'train_losses': tr_losses,
            'val_losses':   vl_losses,
        }
        all_results.append(result)
        with open(results_file, 'w') as _f:
            json.dump(all_results, _f, indent=2)

    print(f'\nDone. Results saved → {results_file}')
    best_row = max(all_results, key=lambda r: r['best_valid_acc'])
    print(f"Best config : {best_row['config']['name']}")
    print(f"Val Acc     : {best_row['best_valid_acc']*100:.2f}%")
    return all_results, best_row


print("t4_grid_search() and init_ft_embeddings() defined.")


In [ ]:
# ── Task 4a: BiGRU grid search — 108 configs (3 pooling × 3 lr × 4 hid × 3 drop) ──

t4_bigru_results, t4_bigru_best = t4_grid_search(
    arch_name      = 'BiGRU',
    build_model_fn = lambda hid, drop, pool: BiRNN(
        VOCAB_SIZE, EMBEDDING_DIM, hid, OUTPUT_DIM,
        n_layers=2, dropout=drop, pad_idx=PAD_IDX, rnn_type='GRU', pooling=pool
    ),
    pooling_options = ['last', 'mean', 'max'],
    results_file    = os.path.join(DATA_DIR, 'Task4_BiGRU_HyperparameterTuning.json'),
)

T4_BIGRU_BEST_POOL = t4_bigru_best['config']['pooling']
T4_BIGRU_BEST_LR   = t4_bigru_best['config']['lr']
T4_BIGRU_BEST_HID  = t4_bigru_best['config']['hidden_dim']
T4_BIGRU_BEST_DROP = t4_bigru_best['config']['dropout']
T4_BIGRU_BEST_WD   = t4_bigru_best['config']['wd']

print(f"\nBest BiGRU config:")
print(f"  pool={T4_BIGRU_BEST_POOL}  lr={T4_BIGRU_BEST_LR}  "
      f"hid={T4_BIGRU_BEST_HID}  drop={T4_BIGRU_BEST_DROP}")


In [ ]:
# ── Task 4b: BiLSTM grid search — 108 configs ─────────────────────────────────

t4_bilstm_results, t4_bilstm_best = t4_grid_search(
    arch_name      = 'BiLSTM',
    build_model_fn = lambda hid, drop, pool: BiRNN(
        VOCAB_SIZE, EMBEDDING_DIM, hid, OUTPUT_DIM,
        n_layers=2, dropout=drop, pad_idx=PAD_IDX, rnn_type='LSTM', pooling=pool
    ),
    pooling_options = ['last', 'mean', 'max'],
    results_file    = os.path.join(DATA_DIR, 'Task4_BiLSTM_HyperparameterTuning.json'),
)

T4_BILSTM_BEST_POOL = t4_bilstm_best['config']['pooling']
T4_BILSTM_BEST_LR   = t4_bilstm_best['config']['lr']
T4_BILSTM_BEST_HID  = t4_bilstm_best['config']['hidden_dim']
T4_BILSTM_BEST_DROP = t4_bilstm_best['config']['dropout']
T4_BILSTM_BEST_WD   = t4_bilstm_best['config']['wd']

print(f"\nBest BiLSTM config:")
print(f"  pool={T4_BILSTM_BEST_POOL}  lr={T4_BILSTM_BEST_LR}  "
      f"hid={T4_BILSTM_BEST_HID}  drop={T4_BILSTM_BEST_DROP}")


In [ ]:
# ── Task 4c: TextCNN grid search — 72 configs (2 pooling × 3 lr × 4 filters × 3 drop) ──
# 'last' pooling is undefined for CNNs (no sequential hidden states)

t4_cnn_results, t4_cnn_best = t4_grid_search(
    arch_name      = 'TextCNN',
    build_model_fn = lambda hid, drop, pool: TextCNN(
        VOCAB_SIZE, EMBEDDING_DIM, n_filters=hid, output_dim=OUTPUT_DIM,
        dropout=drop, pad_idx=PAD_IDX, pooling=pool
    ),
    pooling_options = ['max', 'mean'],   # no 'last' for CNN
    results_file    = os.path.join(DATA_DIR, 'Task4_TextCNN_HyperparameterTuning.json'),
)

T4_CNN_BEST_POOL = t4_cnn_best['config']['pooling']
T4_CNN_BEST_HID  = t4_cnn_best['config']['hidden_dim']   # maps to n_filters
T4_CNN_BEST_LR   = t4_cnn_best['config']['lr']
T4_CNN_BEST_DROP = t4_cnn_best['config']['dropout']
T4_CNN_BEST_WD   = t4_cnn_best['config']['wd']

print(f"\nBest TextCNN config:")
print(f"  pool={T4_CNN_BEST_POOL}  lr={T4_CNN_BEST_LR}  "
      f"n_filters={T4_CNN_BEST_HID}  drop={T4_CNN_BEST_DROP}")


In [ ]:
# ── Task 4: Retrain best config per architecture for full N_EPOCHS ─────────────

os.makedirs(MODEL_DIR, exist_ok=True)

def retrain_best_t4(arch_name, build_model_fn, best_row, save_path):
    cfg = best_row['config']
    print(f"\n{'='*60}")
    print(f"Retraining best {arch_name}: {cfg['name']}")
    print(f"{'='*60}")
    torch.manual_seed(SEED)
    m = build_model_fn(cfg['hidden_dim'], cfg['dropout'], cfg['pooling']).to(device)
    init_ft_embeddings(m)
    opt = optim.Adam(m.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    print(f"  Parameters: {count_parameters(m):,}")
    run_training(m, train_iterator, valid_iterator, opt, criterion,
                 n_epochs=50, save_path=save_path, patience=5)
    m.load_state_dict(torch.load(save_path))
    _, v  = evaluate_epoch(m, valid_iterator, criterion)
    _, te = evaluate_epoch(m, test_iterator,  criterion)
    print(f"  Valid: {v*100:.2f}%   Test: {te*100:.2f}%")
    return m, v, te


model_bigru, valid_acc_bigru, test_acc_bigru = retrain_best_t4(
    'BiGRU',
    lambda hid, drop, pool: BiRNN(
        VOCAB_SIZE, EMBEDDING_DIM, hid, OUTPUT_DIM,
        n_layers=2, dropout=drop, pad_idx=PAD_IDX, rnn_type='GRU', pooling=pool),
    t4_bigru_best,
    os.path.join(MODEL_DIR, 'task4-bigru-best.pt')
)

model_bilstm, valid_acc_bilstm, test_acc_bilstm = retrain_best_t4(
    'BiLSTM',
    lambda hid, drop, pool: BiRNN(
        VOCAB_SIZE, EMBEDDING_DIM, hid, OUTPUT_DIM,
        n_layers=2, dropout=drop, pad_idx=PAD_IDX, rnn_type='LSTM', pooling=pool),
    t4_bilstm_best,
    os.path.join(MODEL_DIR, 'task4-bilstm-best.pt')
)

model_cnn, valid_acc_cnn, test_acc_cnn = retrain_best_t4(
    'TextCNN',
    lambda hid, drop, pool: TextCNN(
        VOCAB_SIZE, EMBEDDING_DIM, n_filters=hid, output_dim=OUTPUT_DIM,
        dropout=drop, pad_idx=PAD_IDX, pooling=pool),
    t4_cnn_best,
    os.path.join(MODEL_DIR, 'task4-cnn-best.pt')
)


In [ ]:
# ── Task 4: Architecture Comparison ───────────────────────────────────────────

print("=" * 78)
print("Task 4 — Architecture Comparison  (FastText 300d, grid search per arch)")
print("-" * 78)
print(f"{'Model':<35} {'Best Pooling':>13} {'Valid Acc':>10} {'Test Acc':>10}")
print("-" * 78)
print(f"{'Baseline: RNN (Task 2)':<35} {'final h_T':>13} "
      f"{valid_acc_t2*100:>9.2f}% {test_acc_t2*100:>9.2f}%")
print(f"{'BiGRU  (best config)':<35} {T4_BIGRU_BEST_POOL:>13} "
      f"{valid_acc_bigru*100:>9.2f}% {test_acc_bigru*100:>9.2f}%")
print(f"{'BiLSTM (best config)':<35} {T4_BILSTM_BEST_POOL:>13} "
      f"{valid_acc_bilstm*100:>9.2f}% {test_acc_bilstm*100:>9.2f}%")
print(f"{'TextCNN (best config)':<35} {T4_CNN_BEST_POOL:>13} "
      f"{valid_acc_cnn*100:>9.2f}% {test_acc_cnn*100:>9.2f}%")
print("=" * 78)
print(f"\nGrid search results saved to:")
print(f"  .data/Task4_BiGRU_HyperparameterTuning.json  ({len(t4_bigru_results)} configs)")
print(f"  .data/Task4_BiLSTM_HyperparameterTuning.json ({len(t4_bilstm_results)} configs)")
print(f"  .data/Task4_TextCNN_HyperparameterTuning.json ({len(t4_cnn_results)} configs)")


---
## Task 5: Ensembling

We combine the four best models — the Task 2 RNN baseline and the three Task 4 architectures — using three strategies:

| Strategy | How it works |
|---|---|
| **Majority Voting** | Each model casts a hard vote; plurality class wins |
| **Weighted Average** | Softmax probabilities averaged with weights ∝ softmax(val accuracies) |
| **Learned FFN Ensembler** | A 2-layer FFN trained on validation-set logits to learn optimal combination |

All base models are **frozen**; only the FFN ensembler has trainable parameters. The FFN is trained on validation-set predictions and evaluated on the held-out test set.


In [ ]:
# ── Task 5: Collect logits and labels from all four models ────────────────────

def get_logits_and_labels(model, iterator):
    """Run inference and return (logits [N, C], labels [N]) as CPU tensors."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in iterator:
            text, lengths = batch.text
            logits = model(text, lengths)
            all_logits.append(logits.cpu())
            all_labels.append(batch.label.cpu())
    return torch.cat(all_logits, dim=0), torch.cat(all_labels, dim=0)

def acc(logits, labels):
    return (logits.argmax(dim=1) == labels).float().mean().item()

print("Collecting logits from all 4 models on val and test sets...")

rnn_val,    val_labels   = get_logits_and_labels(model_t2,     valid_iterator)
bigru_val,  _            = get_logits_and_labels(model_bigru,  valid_iterator)
bilstm_val, _            = get_logits_and_labels(model_bilstm, valid_iterator)
cnn_val,    _            = get_logits_and_labels(model_cnn,    valid_iterator)

rnn_test,    test_labels  = get_logits_and_labels(model_t2,     test_iterator)
bigru_test,  _            = get_logits_and_labels(model_bigru,  test_iterator)
bilstm_test, _            = get_logits_and_labels(model_bilstm, test_iterator)
cnn_test,    _            = get_logits_and_labels(model_cnn,    test_iterator)

val_logits_list  = [rnn_val,  bigru_val,  bilstm_val,  cnn_val]
test_logits_list = [rnn_test, bigru_test, bilstm_test, cnn_test]

print("\nIndividual model accuracies (sanity check):")
print(f"  {'Model':<10}  {'Valid':>8}  {'Test':>8}")
for name, vl, tl in zip(['RNN', 'BiGRU', 'BiLSTM', 'TextCNN'],
                         val_logits_list, test_logits_list):
    print(f"  {name:<10}  {acc(vl, val_labels)*100:>7.2f}%  {acc(tl, test_labels)*100:>7.2f}%")


In [ ]:
# ── Task 5a: Majority Voting Ensemble ────────────────────────────────────────
# Each model casts a hard vote for its top class; plurality wins.
# Ties are broken by the model with the highest validation accuracy.

def voting_ensemble(logits_list, labels):
    preds = torch.stack([l.argmax(dim=1) for l in logits_list], dim=1)  # [N, M]
    # torch.mode returns the most frequent value along dim=1
    voted, _ = torch.mode(preds, dim=1)   # [N]
    return (voted == labels).float().mean().item()

vote_val  = voting_ensemble(val_logits_list,  val_labels)
vote_test = voting_ensemble(test_logits_list, test_labels)

print(f"Majority Voting Ensemble")
print(f"  Valid Acc : {vote_val*100:.2f}%")
print(f"  Test  Acc : {vote_test*100:.2f}%")


In [ ]:
# ── Task 5b: Weighted Average Ensemble ───────────────────────────────────────
# Weights = softmax of individual validation accuracies (sharpened with temperature).
# The model with highest val acc contributes more to the combined probability.

val_accs = torch.tensor([acc(l, val_labels) for l in val_logits_list])
# Temperature scaling: lower temperature → sharper weight distribution
TEMPERATURE = 0.05
weights = torch.softmax(val_accs / TEMPERATURE, dim=0)

print("Ensemble weights (softmax of val accs, T=0.05):")
for name, w, va in zip(['RNN', 'BiGRU', 'BiLSTM', 'TextCNN'], weights, val_accs):
    print(f"  {name:<10}  val_acc={va*100:.2f}%  weight={w.item():.4f}")


def weighted_avg_ensemble(logits_list, weights, labels):
    # Stack softmax probs: [M, N, C], weight each model, sum over M
    probs    = torch.stack([torch.softmax(l, dim=1) for l in logits_list], dim=0)  # [M, N, C]
    w        = weights.view(-1, 1, 1)                                               # [M, 1, 1]
    combined = (probs * w).sum(dim=0)                                               # [N, C]
    return (combined.argmax(dim=1) == labels).float().mean().item()


wavg_val  = weighted_avg_ensemble(val_logits_list,  weights, val_labels)
wavg_test = weighted_avg_ensemble(test_logits_list, weights, test_labels)

print(f"\nWeighted Average Ensemble")
print(f"  Valid Acc : {wavg_val*100:.2f}%")
print(f"  Test  Acc : {wavg_test*100:.2f}%")


In [ ]:
# ── Task 5c: Learned FFN Ensembler (Stacking) ────────────────────────────────
# Meta-learner input : concatenated softmax probs from all 4 models → [4 × C] = 24 features
# Meta-learner output: 6-class logits
# Training set       : validation predictions (standard stacking protocol)
# Evaluation set     : test set (unbiased — never seen during meta-training)

class FFNEnsembler(nn.Module):
    """2-layer MLP that learns to combine predictions from multiple models."""
    def __init__(self, n_models, n_classes, hidden=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_models * n_classes, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, n_classes)
        )
    def forward(self, x):
        return self.net(x)


N_MODELS  = 4
N_CLASSES = OUTPUT_DIM   # 6

def build_meta_features(logits_list):
    """Concatenate softmax probabilities from all models → [N, M*C]."""
    return torch.cat([torch.softmax(l, dim=1) for l in logits_list], dim=1)

meta_val_x  = build_meta_features(val_logits_list).to(device)
meta_test_x = build_meta_features(test_logits_list).to(device)
meta_val_y  = val_labels.to(device)
meta_test_y = test_labels.to(device)

# ── Train meta-learner ────────────────────────────────────────────────────────
torch.manual_seed(SEED)
ffn      = FFNEnsembler(N_MODELS, N_CLASSES, hidden=64, dropout=0.3).to(device)
ffn_opt  = optim.Adam(ffn.parameters(), lr=1e-3, weight_decay=1e-4)
ffn_crit = nn.CrossEntropyLoss()

best_val_loss_ffn = float('inf')
best_ffn_state    = None

for epoch in range(300):
    ffn.train()
    ffn_opt.zero_grad()
    loss = ffn_crit(ffn(meta_val_x), meta_val_y)
    loss.backward()
    ffn_opt.step()

    if loss.item() < best_val_loss_ffn:
        best_val_loss_ffn = loss.item()
        best_ffn_state    = {k: v.clone() for k, v in ffn.state_dict().items()}

ffn.load_state_dict(best_ffn_state)
ffn.eval()
with torch.no_grad():
    ffn_val_acc  = (ffn(meta_val_x).argmax(1)  == meta_val_y).float().mean().item()
    ffn_test_acc = (ffn(meta_test_x).argmax(1) == meta_test_y).float().mean().item()

print(f"Learned FFN Ensembler  (trained on val predictions, eval on test)")
print(f"  Valid Acc : {ffn_val_acc*100:.2f}%  (meta-training set — optimistic)")
print(f"  Test  Acc : {ffn_test_acc*100:.2f}%  (held-out — unbiased)")


In [ ]:
# ── Task 5: Final Ensembling Results ─────────────────────────────────────────

print("=" * 70)
print("Task 5 — Ensembling Results")
print("-" * 70)
print(f"{'Model / Strategy':<38} {'Valid Acc':>10} {'Test Acc':>10}")
print("-" * 70)
print("  Base models:")
print(f"  {'RNN  (Task 2 baseline)':<36} {valid_acc_t2*100:>9.2f}% {test_acc_t2*100:>9.2f}%")
print(f"  {'BiGRU   (Task 4 best)':<36} {valid_acc_bigru*100:>9.2f}% {test_acc_bigru*100:>9.2f}%")
print(f"  {'BiLSTM  (Task 4 best)':<36} {valid_acc_bilstm*100:>9.2f}% {test_acc_bilstm*100:>9.2f}%")
print(f"  {'TextCNN (Task 4 best)':<36} {valid_acc_cnn*100:>9.2f}% {test_acc_cnn*100:>9.2f}%")
print("-" * 70)
print("  Ensemble strategies:")
print(f"  {'Majority Voting':<36} {vote_val*100:>9.2f}% {vote_test*100:>9.2f}%")
print(f"  {'Weighted Average (softmax val acc)':<36} {wavg_val*100:>9.2f}% {wavg_test*100:>9.2f}%")
print(f"  {'Learned FFN Ensembler':<36} {ffn_val_acc*100:>9.2f}% {ffn_test_acc*100:>9.2f}%  *")
print("=" * 70)
print("* FFN trained on val predictions; test acc is unbiased.")


---
## Final Summary of All Results

In [ ]:
print("=" * 78)
print("COMPLETE RESULTS SUMMARY — CE7455 Assignment 1")
print("=" * 78)
print(f"{'Task':<48} {'Valid Acc':>10} {'Test Acc':>10}")
print("-" * 78)
print(f"{'T1: RNN+Packed+Dropout (best hyper, rand init)':<48} "
      f"{valid_acc_t1*100:>9.2f}% {test_acc_t1*100:>9.2f}%")
print(f"{'T2: + FastText 300d (best hyper)':<48} "
      f"{valid_acc_t2*100:>9.2f}% {test_acc_t2*100:>9.2f}%")
print("-" * 78)
print("T3: Output Pooling Strategies (FastText, T2 best params)")
for name in ['MeanPool', 'MaxPool', 'AttnPool']:
    r = t3_results[name]
    marker = '  *best*' if name == best_t3_name else ''
    print(f"  {'T3-'+name:<46} {r['valid']*100:>9.2f}% {r['test']*100:>9.2f}%{marker}")
print("-" * 78)
print("T4: Architecture + Grid Search (FastText, pooling per arch)")
print(f"  {'T4 BiGRU  (pool='+T4_BIGRU_BEST_POOL+')':<46} "
      f"{valid_acc_bigru*100:>9.2f}% {test_acc_bigru*100:>9.2f}%")
print(f"  {'T4 BiLSTM (pool='+T4_BILSTM_BEST_POOL+')':<46} "
      f"{valid_acc_bilstm*100:>9.2f}% {test_acc_bilstm*100:>9.2f}%")
print(f"  {'T4 TextCNN(pool='+T4_CNN_BEST_POOL+')':<46} "
      f"{valid_acc_cnn*100:>9.2f}% {test_acc_cnn*100:>9.2f}%")
print("-" * 78)
print("T5: Ensembling (T2 + T4 best models)")
print(f"  {'T5a Majority Voting':<46} {vote_val*100:>9.2f}% {vote_test*100:>9.2f}%")
print(f"  {'T5b Weighted Average (softmax val acc)':<46} {wavg_val*100:>9.2f}% {wavg_test*100:>9.2f}%")
print(f"  {'T5c Learned FFN Ensembler':<46} {ffn_val_acc*100:>9.2f}% {ffn_test_acc*100:>9.2f}%")
print("=" * 78)
